# All Codes - Data Preparation Script

Imports And Filter Warnings

In [18]:
#Please pip install the below libraries from terminal before running.

import json
import pandas as pd
import os
import math
import warnings
import random
import re
import glob
from datetime import datetime
warnings.filterwarnings("ignore")

File Paths

In [19]:
#Please create the relevant folders before running the scripts. Refer to the ReadMe file for more details 

input_directory = r'C:\Users\singh\Downloads\EAS508 Project'
state = r'\California'
output_directory = input_directory

## Reddit Processing Scripts

Formatting Script - JSON to CSV

In [ ]:
'''
Below Script converts JSONL Files (posts and comments) scraped from reddit, or downloaded through the PushShift Dumps into csv format
for further processing. Please refer to the ReadMe file for the relevant changes before running this script
'''

def jsonl_to_excel(input_dir, output_dir):
    for filename in os.listdir(input_dir):
        if filename.endswith('.jsonl'):
            input_jsonl_file = os.path.join(input_dir, filename)
            output_excel_file = os.path.join(output_dir, filename.replace('.jsonl', '_body.xlsx'))
            
            if os.path.exists(output_excel_file):
                print(f"File {output_excel_file} already exists. Skipping conversion.")
                continue
            
            with open(input_jsonl_file, 'r', encoding='utf-8') as jsonl_file:
                data = [json.loads(line) for line in jsonl_file]
            
            df = pd.DataFrame(data)
            
            if 'body' not in df.columns:
                print(f"Error: 'body' column not found in {input_jsonl_file}. Skipping file.")
                continue
            
            body_column = df[['body']]
            
            max_rows_per_sheet = 900000
            num_sheets = math.ceil(len(body_column) / max_rows_per_sheet)

            with pd.ExcelWriter(output_excel_file, engine='xlsxwriter') as writer:
                for i in range(num_sheets):
                    start_row = i * max_rows_per_sheet
                    end_row = min((i + 1) * max_rows_per_sheet, len(body_column))
                    sheet_name = f'Sheet_{i+1}'
                    body_column.iloc[start_row:end_row].to_excel(writer, sheet_name=sheet_name, index=False)
            
            print(f"Converted {input_jsonl_file} to {output_excel_file}")
            print(f"Total rows: {len(body_column)}, Sheets created: {num_sheets}")

# File paths (Please do not change this, follow the ReadMe directions)
input_subfolder = r'\Reddit Data And PreProcessing\Reddit Raw Files'
output_subfolder = r'\\Reddit Data And PreProcessing\Reddit CSV Converted'
jsonl_path = input_directory + input_subfolder + state
csv_converted_path = output_directory + output_subfolder + state

#Handles errors and created in case output directly is named improperly
os.makedirs(csv_converted_path, exist_ok=True)

# Execute the conversion
jsonl_to_excel(jsonl_path, csv_converted_path)

Filtering Script

In [ ]:
'''
Below Script filters the comments and posts stored in the csv file (output from the Formatting Script) for certain crime related keywords
It also merges all files within the same state directory (from different subreddits/cities) to a single file
Please refer to the ReadMe file for the relevant changes before running this script
'''

# Define the crime phrases and crime types
CRIME_PHRASES = [
    "crime", "violent", "violence",
    "theft", "robbery", "assault", "suspicious activity", 
    "vandalism", "shooting", "murder", "burglary", 
    "police report", "public safety", "neighborhood safety",
    "breaking and entering", "armed robbery", "crime wave",
    "robbed", "robberies", "guns", "assaulted", 
    "rob", "burglaries",
    "dangerous area", "bad neighborhood", "dangerous street", 
    "stolen", "car theft", "hit and run", "arson", 
    "looting", "harassment", "kidnapping", "human trafficking", "rape", 
    "sexual assault", "drug dealing", "drug bust", 
    "drug trafficking", "drugs", "extortion", "gang activity",
    "home invasion", "illegal firearms", 
    "law enforcement", "police chase", "shootout",
    "shoplifting", "smuggling", "stabbing", 
    "terrorism", "threats", "youth violence",
    "aggravated assault", "attempted murder", 
    "battery", "gang violence", "hate crime", "homicide", 
    "prostitution", "ransom",
    "road rage", "school violence", "serial killer",
    "sexual harassment", "street racing", "trafficking",
    "vehicle theft",
    "abduction", "armed assault", "bribery",
    "death threat", "organized crime", 
    "property damage", "public disturbance", 
    "riot", "self-defense", 
    "theft by deception", "threatening behavior",
    "not safe to walk", "unsafe to walk",
    "insecure area", "unsafe neighborhood"
]

CRIME_TYPES = [
    "armed robbery", "home invasion", "car theft", "drug trafficking",
    "assault and battery", "domestic violence", "sexual assault",
    "homicide investigation", "gang activity", "cybercrime report",
    "fraud alert", "kidnapping incident",
    "burglary in progress", "shoplifting arrest", "vandalism spree",
    "police chase", "shooting reported", "stabbing victim",
    "crime scene investigation", "murder suspect", "drug bust",
    "human trafficking", "identity theft", "money laundering",
    "terrorist attack", "hate crime", "child abuse", "illegal weapons",
    "carjacking incident", "arson investigation", "bank robbery",
    "prison break", "police brutality", "gang-related violence",
    "cyber attack", "embezzlement scheme", "counterfeit operation",
    "drug overdose", "serial killer", "mass shooting", "bomb threat",
    "hostage situation", "illegal gambling", "organized crime",
    "police officer shot", "wanted fugitive", "crime ring busted"
]

def find_exact_matches(text, phrases):
    matches = []
    for phrase in phrases:
        if re.search(r'\b' + re.escape(phrase) + r'\b', text, re.IGNORECASE):
            matches.append(phrase)
    return matches

def process_excel_files(input_dir, output_file):
    all_filtered_rows = []
    
    for filename in os.listdir(input_dir):
        if filename.endswith('.xlsx'):
            file_path = os.path.join(input_dir, filename)
            xls = pd.ExcelFile(file_path)
            
            for sheet_name in xls.sheet_names:
                df = pd.read_excel(xls, sheet_name)
                
                if 'body' not in df.columns:
                    print(f"Error: 'body' column not found in {filename}, sheet {sheet_name}. Skipping.")
                    continue
                
                # Filter rows containing exact matches of crime phrases
                df['matches'] = df['body'].apply(lambda x: find_exact_matches(str(x), CRIME_PHRASES))
                filtered_df = df[df['matches'].apply(len) > 0]
                
                # Add crime type column
                filtered_df['crime_type'] = filtered_df['body'].apply(lambda x: ', '.join(find_exact_matches(str(x), CRIME_TYPES)) or ', '.join(find_exact_matches(str(x), CRIME_PHRASES)) or "Unknown")
                
                all_filtered_rows.append(filtered_df)
    
    # Combine all filtered rows
    result_df = pd.concat(all_filtered_rows, ignore_index=True)
    
    # Remove the 'matches' column
    result_df = result_df.drop('matches', axis=1)

    result_df['Location'] = state[1:] 
    
    # Save to Excel file with multiple sheets if necessary
    max_rows_per_sheet = 900000
    with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
        for i in range(0, len(result_df), max_rows_per_sheet):
            sheet_name = f'Sheet_{i//max_rows_per_sheet + 1}'
            result_df.iloc[i:i+max_rows_per_sheet].to_excel(writer, sheet_name=sheet_name, index=False)
    
    print(f"Processed {len(all_filtered_rows)} files.")
    print(f"Total filtered rows: {len(result_df)}")
    print(f"Results saved to {output_file}")


# File paths
input_subfolder = r'\\Reddit Data And PreProcessing\Reddit CSV Converted'
output_subfolder = r'\Reddit Data And PreProcessing\Reddit Filtered'
csv_converted_path = input_directory + input_subfolder + state
Filtered_data_path = output_directory + output_subfolder + state + '.xlsx'


# Ensure the output directory exists
os.makedirs(os.path.dirname(Filtered_data_path), exist_ok=True)


# Execute the processing
process_excel_files(csv_converted_path, Filtered_data_path)

Cleaning Script

In [ ]:
'''
Below Script cleans the 'filtered' data files by removing certain keywords that appear repeatedly across all state files
but are irrelevant as indicators for crime.
Please refer to the ReadMe file for the relevant changes before running this script
'''

# Words to remove (not exact matches)
words_to_remove = ['Adams', 'Battery Park', 'Biden', 'Busch', 'Caylee', 'Charles', 'Election', 'England',
                'Harris', 'India', 'Ireland', 'Jimmy', 'NFL', 'New Zealand', 'Paula', 'Quiet Riot',
                'Schneider', 'True Crime', 'Trump', 'UAE', 'US', 'Zimmerman', 'alien', 'anatomy',
                'anti-gun', 'baby', 'battery', 'bizarre', 'borrow', 'child abuse', 'conservat',
                'craigs',  'death', 'democracy', 'democrat', 'democrats', 'dogs',
                'domestic violence', 'drugs',  'economics', 'enforcement', 'fantasies', 'felony',
                'film', 'football', 'fun ride',  'garbage', 'gun control', 'gun ownership', 'guns', 'hitler',
                'http', 'identity',  'imperialism', 'income', 'insurance', 'is a crime', 'jackie', 'joke',
                'kidding', 'kids',  'landlord', 'lawyer', 'learn', 'lease', 'left wing', 'lmao', 'marraige',
                'mayor', 'med',  'mortality', 'news', 'not a crime', 'penalty', 'picture', 'pictures', 'poetry',
                'police report', 'policies', 'policy', 'politics', 'porn', 'prank', 'prison', 'pro-gun',
                'prostitution', 'protest', 'radio show', 'rape', 'reddit', 'republic', 'republican',
                'republicans', 'ride', 'right wing', 'rights', 'riot', 'road rage', 'scam', 'serve',
                'sherrif', 'shooting', 'shooting range', 'show', 'sketch', 'slavery', 'snl', 'spare key',
                'strict law', 'strict laws', 'tagged', 'take the', 'taxpayer', 'television', 'terrorism',
                'thread', 'threats', 'trafficking', 'trees', 'ufo', 'vaccin', 'violence', 'violent', 'war',  'years'
                'lol', 'covid', 'code', 'where', 'what', 'homicide', 'charge', 'legal', 'penalty', 'season', 'fake', 'pandemic'
                'prosecute', 'ass', 'law', 'article', 'post', 'troll', 'arrest', 'vote', 'elect', 'serial', 'GTA', 'resign', 'Pay'
                'copyright', '90s', '80s', 'tik tok', 'trans', 'era', 'legislation', 'term', 'coldplay', 'taylor', 'sing', 'concert',
                'mass', 'food', 'Asian', 'fries', 'attempted', 'dog', 'cat', 'animal', 'Grand Theft Auto', 'fuck', 'Lenin', 'capitalism'
                'communism', 'batman', 'suckers', 'LOL', 'Lol!', 'Lol?', 'tax', 'dick', 'superman', '911', 'cops', 'cop', 'replica', 'r/',
                'Carthy', 'artist', 'talent', 'shneider', 'zombies', 'Grankow', 'karma', 'Liefie', 'mystery',
                'office', 'shit', 'caught', 'sex', 'toy', 'valor', 'hills', 'woods'
]


def remove_rows_with_words(input_file, output_file, words_to_remove):
    # Read the input Excel file
    df = pd.read_excel(input_file)
    
    # Function to check if any word in the list is in the text
    def contains_word(text, word_list):
        return any(word.lower() in str(text).lower() for word in word_list)
    
    # Filter out rows containing any of the specified words
    filtered_df = df[~df['body'].apply(lambda x: contains_word(x, words_to_remove))]
    
    # Save the filtered DataFrame to a new Excel file
    filtered_df.to_excel(output_file, index=False)
    
    print(f"Original row count: {len(df)}")
    print(f"Filtered row count: {len(filtered_df)}")
    print(f"Removed {len(df) - len(filtered_df)} rows")
    print(f"Results saved to {output_file}")


# File paths
input_subfolder = r'\Reddit Data And PreProcessing\Reddit Filtered'
output_subfolder = r'\Reddit Data And PreProcessing\Reddit Cleaned'
Filtered_data_path = input_directory + input_subfolder + state + '.xlsx'
Cleaned_data_path = output_directory + output_subfolder + state + '.xlsx'


# Ensure the output directory exists
os.makedirs(os.path.dirname(Cleaned_data_path), exist_ok=True)


# Execute the filtering
remove_rows_with_words(Filtered_data_path, Cleaned_data_path, words_to_remove)

Concatenation Script - Merge and Stratify Data

In [ ]:
'''
Below Script combines the cleaned data files for multiple states and stratifies the data by 
considering a set number of rows randomly from each data file
Please refer to the ReadMe file for the relevant changes before running this script
'''

def select_random_rows_from_files(input_folder, output_file, num_rows=2000):
    combined_data = []
    
    for filename in os.listdir(input_folder):
        if filename.endswith('.xlsx'):
            file_path = os.path.join(input_folder, filename)
            df = pd.read_excel(file_path)
            if len(df) >= num_rows:
                selected_rows = df.sample(n=num_rows, random_state=1)
                combined_data.append(selected_rows)
            else:
                print(f'File {filename} has less than {num_rows} rows. Skipping this file.')
    
    if combined_data:
        final_data = pd.concat(combined_data, ignore_index=True)
        final_data.to_excel(output_file, index=False)
        print(f'Selected rows saved to {output_file}')
    else:
        print('No files were processed.')


# File paths
input_subfolder = r'\Reddit Data And PreProcessing\Reddit Cleaned'
output_subfolder = r'\Reddit Data And PreProcessing\Reddit Merged'
Cleaned_data_path = input_directory + input_subfolder
Merged_data_path = output_directory + output_subfolder


# Ensure the output directory exists
os.makedirs(Merged_data_path, exist_ok=True)
Merged_file_path = os.path.join(Merged_data_path, 'Merged.xlsx')

# Execute the merging
select_random_rows_from_files(Cleaned_data_path, Merged_file_path, num_rows = 500)


## Twitter Data Processing Scripts

Crime Tagging Script

In [ ]:
'''
Below Script converts JSONL Files (posts and comments) scraped from reddit, or downloaded through the PushShift Dumps into csv format
for further processing. 
Please refer to the ReadMe file for the relevant changes before running this script
'''

CRIME_PHRASES = [
    "armed_robbery", "home_invasion", "car_theft", "drug_trafficking",
    "assault_and_battery", "domestic_violence", "sexual_assault",
    "homicide_investigation", "gang_activity", "cybercrime_report",
    "fraud_alert", "missing_person_case", "kidnapping_incident",
    "burglary_in_progress", "shoplifting_arrest", "vandalism_spree",
    "police_chase", "shooting_reported", "stabbing_victim",
    "crime_scene_investigation", "murder_suspect", "drug_bust",
    "human_trafficking", "identity_theft", "money_laundering",
    "terrorist_attack", "hate_crime", "child_abuse", "illegal_weapons",
    "carjacking_incident", "arson_investigation", "bank_robbery",
    "prison_break", "police_brutality", "gang-related_violence",
    "cyber_attack", "embezzlement_scheme", "counterfeit_operation",
    "drug_overdose", "serial_killer", "mass_shooting", "bomb_threat",
    "hostage_situation", "illegal_gambling", "organized_crime",
    "police_officer_shot", "wanted_fugitive", "crime_ring_busted"
]

def determine_category(filename):
    filename_lower = filename.lower()
    for phrase in CRIME_PHRASES:
        if phrase.lower() in filename_lower:
            return phrase
    return 'Other'

def process_csv_files(folder_path):
    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            file_path = os.path.join(folder_path, filename)
            category = determine_category(filename)
            df = pd.read_csv(file_path)
            df['Category'] = category
            df.to_csv(file_path, index=False)
            print(f"Processed {filename}, assigned category: {category}")

# File paths
input_subfolder = r'\Twitter Data and Preprocessing\Tweets Data'
Data_path = input_directory + input_subfolder


# Run the processing function
process_csv_files(Data_path)

Merging Script

In [ ]:
'''
Below Script merges the individual scraped files from reddit to form a single merged file for further processing. 
Please refer to the ReadMe file for the relevant changes before running this script
'''

def merge_csv_files(input_folder, output_file):
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
    
    if not csv_files:
        print("No CSV files found in the specified folder.")
        return
    
    dfs = []
    
    for file in csv_files:
        df = pd.read_csv(file)
        dfs.append(df)
        print(f"Read: {file}")
    
    combined_df = pd.concat(dfs, ignore_index=True)
    
    combined_df.to_csv(output_file, index=False)
    print(f"Merged CSV file saved as: {output_file}")
    print(f"Total rows in merged file: {len(combined_df)}")


# File paths
input_subfolder = r'\Twitter Data and Preprocessing\Tweets Data'
output_subfolder = r'\Twitter Data and Preprocessing\Twitter Data Cleaned and Merged'
Data_path = input_directory + input_subfolder
Merged_data_path = output_directory + output_subfolder
os.makedirs(Merged_data_path, exist_ok=True)
Merged_file_path = os.path.join(Merged_data_path, 'Merged.csv')

# Run the merging function
merge_csv_files(Data_path, Merged_file_path)

## Historical Data Processing

Loading historical data

In [20]:
'''
Below script the imports data from .csv files from individual police websites for all five states.
Please refer to the ReadMe file for the relevant changes before running this script
'''

California_Data = pd.read_csv(input_directory + r'\Historical Data and Preprocessing\Raw Data Files\California_Crime_Data.csv')
Illinois_Data = pd.read_csv(input_directory + r'\Historical Data and Preprocessing\Raw Data Files\Illinois_Crime_Data.csv')
Texas_Data = pd.read_csv(input_directory + r'\Historical Data and Preprocessing\Raw Data Files\Texas_Crime_Data.csv')
Washington_Data = pd.read_csv(input_directory + r'\Historical Data and Preprocessing\Raw Data Files\Washington_Crime_Data.csv')
New_York = pd.read_csv(input_directory + r'\Historical Data and Preprocessing\Raw Data Files\New_York_Crime_Data.csv')

Historical data preprocessing - merging of data files

In [21]:
'''
Below script performs data preprocessing by changing date column into correct format, tagging states, retaines relevant columns and returns a merged file. 
Please refer to the ReadMe file for the relevant changes before running this script
'''

dfs = pd.DataFrame()
df_1 = California_Data.copy()
df_1['date'] = pd.to_datetime(df_1['DATE OCC']).dt.strftime('%m-%Y')
df_1['crime_type'] = df_1['Crm Cd Desc']
df_1['state'] = 'California'
df_1['state'] = df_1['state'].astype(str)
df_1['data'] = df_1['date'].astype(str)
df_1['crime_type'] = df_1['crime_type'].astype(str)
df_1 = df_1[['date', 'crime_type', 'state']].dropna()
df_1 = df_1.head(200000)
dfs = pd.concat([dfs,df_1], ignore_index = True)

df = Texas_Data.copy()
df['date'] = df['Month1 of Occurence'].astype(str).str.zfill(2) + '-' + df['Year1 of Occurrence'].astype(str)
df['crime_type'] = df['Type of Incident']
df['state'] = 'Texas'
df['state'] = df['state'].astype(str)
df['data'] = df['date'].astype(str)
df['crime_type'] = df['crime_type'].astype(str)
df = df[['date', 'crime_type', 'state']].dropna()
df = df.head(200000)
dfs = pd.concat([dfs,df], ignore_index = True)

df = New_York.copy()
df['month_year'] = df['cmplnt_fr_dt'].str[0:11].astype(str)
df['date'] = df['month_year'].str[5:7].astype(str) + "-" + df['month_year'].str[0:4].astype(str)
df['crime_type'] = df['ofns_desc']
df['state'] = 'New York'
df['state'] = df['state'].astype(str)
df['date'] = df['date'].astype(str)
df['crime_type'] = df['crime_type'].astype(str)
df = df[['date', 'crime_type', 'state']].dropna()
df = df.head(200000)
dfs = pd.concat([dfs, df], ignore_index=True)

df = Illinois_Data.copy()
df['date'] = pd.to_datetime(df['Date']).dt.strftime('%m-%Y')
df['crime_type'] = df['Primary Type']
df['state'] = 'Illinois'
df['state'] = df['state'].astype(str)
df['data'] = df['date'].astype(str)
df['crime_type'] = df['crime_type'].astype(str)
df = df[['date', 'crime_type', 'state']].dropna()
df = df.head(200000)
dfs = pd.concat([dfs,df], ignore_index = True)

df = Washington_Data.copy()
df['date'] = pd.to_datetime(df['Offense Start DateTime']).dt.strftime('%m-%Y')
df['crime_type'] = df['Offense']
df['state'] = 'Washington'
df['state'] = df['state'].astype(str)
df['data'] = df['date'].astype(str)
df['crime_type'] = df['crime_type'].astype(str)
df = df[['date', 'crime_type', 'state']].dropna()
df = df.head(200000)
dfs = pd.concat([dfs,df], ignore_index = True)

dfs.to_csv(input_directory + r'\Historical Data and Preprocessing\Merged File\Merged Crime Data.csv', index=False)
print("Processing complete. Results saved to 'Merged Crime Data.csv'")
Merged_Data = pd.read_csv(input_directory + r'\Historical Data and Preprocessing\Merged File\Merged Crime Data.csv')
Merged_Data.shape

Processing complete. Results saved to 'Merged Crime Data.csv'


(125000, 3)

Historical data preprocessing - crime category tagging

In [22]:
'''
Below script adds tags for type of crimes in two layers. 
Please refer to the ReadMe file for the relevant changes before running this script
'''

data = pd.read_csv(input_directory + r'\Historical Data and Preprocessing\Merged File\Merged Crime Data.csv')
crime_categories = {
    'Violent Crimes': ['ASSAULT', 'BATTERY', 'HOMICIDE', 'KIDNAPPING', 'ROBBERY', 'MURDER'],
    'Property Crimes': ['BURGLARY'],
    'Sexual Offenses': ['RAPE', 'SEXUAL'],
    'Miscellaneous Crimes': ['ARSON']
}
def get_primary_category(crime_type):
    for category, keywords in crime_categories.items():
        if any(keyword in crime_type.upper() for keyword in keywords):
            return category
    return ''
def get_secondary_category(crime_type):
    for category, keywords in crime_categories.items():
        for keyword in keywords:
            if keyword in crime_type.upper():
                return keyword
    return 'OTHER'
data['Primary Category'] = data['crime_type'].apply(get_primary_category)
data['Secondary Category'] = data['crime_type'].apply(get_secondary_category)
print(data.head())
data.to_csv(input_directory + r'\Historical Data and Preprocessing\Categorized File\Crime Data Categorized.csv', index=False)

      date                                         crime_type       state  \
0  02-2010                           VIOLATION OF COURT ORDER  California   
1  09-2010  VANDALISM - FELONY ($400 & OVER, ALL CHURCH VA...  California   
2  08-2010                          OTHER MISCELLANEOUS CRIME  California   
3  01-2010                           VIOLATION OF COURT ORDER  California   
4  01-2010                                    RAPE, ATTEMPTED  California   

  Primary Category Secondary Category  
0                               OTHER  
1                               OTHER  
2                               OTHER  
3                               OTHER  
4  Sexual Offenses               RAPE  


Historical data preprocessing - dropping irrelevant crime categories

In [23]:
'''
Below script drops rows with irrelevant crime categories. 
Please refer to the ReadMe file for the relevant changes before running this script
'''

data = pd.read_csv(input_directory + r'\Historical Data and Preprocessing\Categorized File\Crime Data Categorized.csv')
cleaned_data = data[(data['Primary Category'] != 'Financial and Fraud Crimes') & (data['Primary Category'].notnull())]
print(f"Original number of rows: {len(data)}")
print(f"Number of rows after cleaning: {len(cleaned_data)}")
cleaned_data.to_csv(input_directory + r'\Historical Data and Preprocessing\Final Files\Final_Cleaned_Crime_Data.csv', index=False)
print("Cleaned data saved to 'Final_Cleaned_Crime_Data.csv'")


Original number of rows: 125000
Number of rows after cleaning: 31089
Cleaned data saved to 'Final_Cleaned_Crime_Data.csv'


In [24]:
'''
Below script constructs a new column "Months DIfference" based on how long the crime happened, and penalizes severity based on this later
Please refer to the ReadMe file for the relevant changes before running this script
'''

data = pd.read_csv(input_directory + r'\Historical Data and Preprocessing\Final Files\Final_Cleaned_Crime_Data.csv')
data['date'] = pd.to_datetime(data['date'], errors='coerce')
current_date = datetime.now()
data['Months Difference'] = data['date'].apply(lambda x: (current_date.year - x.year) * 12 + (current_date.month - x.month))
data.to_csv(input_directory + r'\Historical Data and Preprocessing\Final Files\Final_Cleaned_Crime_Data.csv', index=False)

Assigning severity scores to crimes

In [26]:
'''
Below script assigns severity score to exisiting crime categories based on weights assigned to each crime. 
Please refer to the ReadMe file for the relevant changes before running this script
'''


data = pd.read_csv(input_directory + r'\Historical Data and Preprocessing\Final Files\Final_Cleaned_Crime_Data.csv')
severity_scores = {
    'Violent Crimes': {
        'ASSAULT': 8,
        'BATTERY': 8,
        'HOMICIDE': 10,
        'KIDNAPPING': 9,
        'ROBBERY': 7.5,
        'MURDER': 10
    },
    'Property Crimes': {
        'BURGLARY': 6.5,
        'THEFT': 5,
        'SHOPLIFTING': 4,
        'VEHICLE': 5
    },
    'Sexual Offenses': {
        'RAPE': 9.5,
        'SEXUAL': 8,
        'INDECENT': 6,
        'LEWD': 6
    },
    'Domestic and Family Crimes': {
        'CHILD': 8,
        'INTIMATE PARTNER': 7,
        'RESTRAINING ORDER': 5
    },
    'Public Order Offenses': {
        'DISTURBING THE PEACE': 3,
        'TRESPASSING': 3,
        'VANDALISM': 4,
        'WEAPONS': 6,
        'TRAFFIC': 2
    },
    'Drug-related Crimes': {
        'DRUG': 5,
        'CANNABIS': 4
    },
    'Miscellaneous Crimes': {
        'ARSON': 6,
        'EXTORTION': 6,
        'STALKING': 5,
        'BOMB': 8,
        'OTHER': 4
    }
}

# Assigning severity score
def assign_severity_score(row):
    primary = row['Primary Category']
    secondary = row['Secondary Category']
    return severity_scores.get(primary, {}).get(secondary, 0)

data['Crime Severity'] = data.apply(assign_severity_score, axis=1)
data.head()
data = data[['date', 'crime_type', 'state', 'Primary Category', 'Secondary Category', 'Months Difference', 'Crime Severity']]

# Save the updated dataset
data.to_csv(input_directory + r'\Historical Data and Preprocessing\Final Files\Final_Crime_Data_With_Severity.csv', index=False)

# Display first few rows of the dataset
print(data.head())


         date                                      crime_type       state  \
0  2010-01-01                                 RAPE, ATTEMPTED  California   
1  2010-01-01                           BURGLARY FROM VEHICLE  California   
2  2010-01-01  ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT  California   
3  2010-01-01  ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT  California   
4  2010-01-01                        BATTERY - SIMPLE ASSAULT  California   

  Primary Category Secondary Category  Months Difference  Crime Severity  
0  Sexual Offenses               RAPE                179             9.5  
1  Property Crimes           BURGLARY                179             6.5  
2   Violent Crimes            ASSAULT                179             8.0  
3   Violent Crimes            ASSAULT                179             8.0  
4   Violent Crimes            ASSAULT                179             8.0  
